# 02 · Data Preprocessing

**Goal:** turn the raw customer table into a clean, model-ready dataset, and create **one fixed train/test split** that every model in notebooks 03–05 uses. That way the model comparison is fair.

**Based on the original project.** The original `1-data-preprocessing.ipynb` by [sinanw](https://github.com/sinanw/ann-bank-customer-retention) dropped the identifier columns, encoded `Gender` and `Geography`, split the data 80/20, and standardised the features using statistics from the training set only. This notebook keeps those steps and changes three things:

| Change | Why |
|---|---|
| **Stratified** split | Keeps the 20.4% churn rate identical in train and test, so test metrics aren't distorted by chance |
| Encoding and scaling wrapped in a reusable scikit-learn `ColumnTransformer` (`src/churn_utils.py`) | The same transformation is applied inside every model pipeline and in the dashboard, and it is fit on training data only, so no test information leaks into training |
| Saved files keep the **original (unscaled) values** plus `CustomerId` | Tree models don't need scaling, the dashboard needs real values, and `CustomerId` lets the business notebook produce a customer-level priority list (it is never used as a model input) |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.churn_utils import (
    BINARY_FEATURES,
    CATEGORICAL_FEATURES,
    FEATURES,
    NUMERIC_FEATURES,
    PROCESSED_DIR,
    RANDOM_STATE,
    TARGET,
    TEST_PATH,
    TEST_SIZE,
    TRAIN_PATH,
    build_preprocessor,
    load_raw_data,
)

df = load_raw_data()
df.shape

(10000, 14)

## 1. Data quality checks

In [2]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique values": df.nunique(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
quality

,dtype,missing,unique values,min,max
Age,int64,0,70,18.00,92.00
Balance,float64,0,6382,0.00,250898.09
CreditScore,int64,0,460,350.00,850.00
CustomerId,int64,0,10000,15565701.00,15815690.00
EstimatedSalary,float64,0,9999,11.58,199992.48
Exited,int64,0,2,0.00,1.00
Gender,str,0,2,NaN,NaN
Geography,str,0,3,NaN,NaN
HasCrCard,int64,0,2,0.00,1.00
IsActiveMember,int64,0,2,0.00,1.00


In [3]:
print("Duplicate rows:        ", df.duplicated().sum())
print("Duplicate customer IDs:", df["CustomerId"].duplicated().sum())
print("Geography values:      ", sorted(df["Geography"].unique()))
print("Gender values:         ", sorted(df["Gender"].unique()))

Duplicate rows:         0
Duplicate customer IDs: 0
Geography values:       ['France', 'Germany', 'Spain']
Gender values:          ['Female', 'Male']


**Result:** no missing values, no duplicates, no invalid category labels. All ranges are plausible (ages 18–92, credit scores 350–850, 1–4 products, binary flags in {0, 1}). No cleaning or imputation is needed.

Two values worth knowing about, which we deliberately **keep**:
* **Zero balances** (36% of customers) are a real customer state, not missing data.
* **Very low `EstimatedSalary`** values (the minimum is about 12) are implausible as real salaries. With no basis for correcting them, and salary showing no association with churn (notebook 01), we leave them as they are.

## 2. Select features

* **Dropped:** `RowNumber` (row index), `Surname` (personal data with no legitimate predictive role). `CustomerId` is kept in the saved files for reporting but **excluded from model inputs**.
* **Kept as model inputs (10 features):**

In [4]:
feature_overview = pd.DataFrame({
    "Feature": FEATURES,
    "Type": (["Categorical"] * len(CATEGORICAL_FEATURES)
             + ["Binary (0/1)"] * len(BINARY_FEATURES)
             + ["Numeric"] * len(NUMERIC_FEATURES)),
})
feature_overview

,Feature,Type
0,Geography,Categorical
1,Gender,Categorical
2,HasCrCard,Binary (0/1)
3,IsActiveMember,Binary (0/1)
4,CreditScore,Numeric
5,Age,Numeric
6,Tenure,Numeric
7,Balance,Numeric
8,NumOfProducts,Numeric
9,EstimatedSalary,Numeric


All 10 features are kept, including those that showed little association with churn in the EDA (credit score, salary, tenure, credit card). Rather than removing them by hand, we let the models and the feature-importance analysis (notebook 05) show how much they contribute.

## 3. Train/test split (stratified 80/20)

The **test set (20% = 2,000 customers) is set aside** and used only once per model for the final evaluation. All model tuning in notebook 03 happens through cross-validation on the training set.

In [5]:
model_data = df[["CustomerId"] + FEATURES + [TARGET]]

train_df, test_df = train_test_split(
    model_data,
    test_size=TEST_SIZE,
    stratify=model_data[TARGET],   # keep the churn rate equal in both sets
    random_state=RANDOM_STATE,     # reproducible split
)

split_summary = pd.DataFrame({
    "Customers": [len(train_df), len(test_df)],
    "Churners": [train_df[TARGET].sum(), test_df[TARGET].sum()],
    "Churn rate": [train_df[TARGET].mean(), test_df[TARGET].mean()],
}, index=["Train", "Test"])
split_summary.style.format({"Churn rate": "{:.1%}"})

,Customers,Churners,Churn rate
Train,8000,1630,20.4%
Test,2000,407,20.3%


Both sets have the same 20.4% churn rate, so the test set is representative of the full customer base.

## 4. Encoding and scaling

The preprocessing steps live in `build_preprocessor()` in `src/churn_utils.py`:

| Feature(s) | Transformation | Reason |
|---|---|---|
| `Geography` | One-hot encoding → `Geography_France`, `Geography_Germany`, `Geography_Spain` | Countries have no natural order, so they can't be coded 0/1/2 |
| `Gender` | Single 0/1 column (`Gender_Male`) | Binary category; one column is enough |
| `HasCrCard`, `IsActiveMember` | Unchanged | Already 0/1 |
| Numeric features | Standardised (mean 0, std 1) | Logistic regression and neural networks are sensitive to feature scale (Balance is in the 100,000s, NumOfProducts is 1–4). Tree models are unaffected |

**Preventing data leakage:** the scaler learns its means and standard deviations from the **training set only**. Inside each model pipeline it is re-fit on each cross-validation fold, so no information from validation or test data reaches the model during training.

Below we fit the preprocessor on the training data to check what the models will receive.

In [6]:
preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(train_df[FEATURES])
transformed = pd.DataFrame(X_train_transformed, columns=preprocessor.get_feature_names_out())

print("Model input shape:", transformed.shape)
transformed.describe().loc[["mean", "std", "min", "max"]].round(2)

Model input shape: (8000, 12)


,Geography_France,Geography_Germany,Geography_Spain,Gender_Male,HasCrCard,IsActiveMember,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary
mean,0.5,0.25,0.25,0.55,0.71,0.51,-0.00,0.00,-0.00,0.00,-0.00,-0.00
std,0.5,0.43,0.43,0.50,0.45,0.50,1.00,1.00,1.00,1.00,1.00,1.00
min,0.0,0.00,0.00,0.00,0.00,0.00,-3.11,-1.99,-1.73,-1.23,-0.91,-1.74
max,1.0,1.00,1.00,1.00,1.00,1.00,2.06,5.04,1.72,2.60,4.25,1.75


The 10 raw features become **12 model inputs** (3 country columns, 1 gender column, 2 binary flags, 6 standardised numeric features). The numeric columns now have mean 0 and standard deviation 1 on the training data, as expected.

## 5. Save the split

In [7]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)
print(f"Saved {TRAIN_PATH.relative_to(PROJECT_ROOT)} ({len(train_df):,} rows)")
print(f"Saved {TEST_PATH.relative_to(PROJECT_ROOT)} ({len(test_df):,} rows)")

Saved data/processed/train.csv (8,000 rows)
Saved data/processed/test.csv (2,000 rows)


**Next:** notebook 03 trains and compares logistic regression, decision tree, random forest and XGBoost on this exact split.